# 🧬 Building a Genomics Research Assistant with Strands, Tools & MCP

This notebook builds on the one-tool code-first Research Assistant you created in Module 1 with the Strands Agents framework. You will reconnect to that agent, grow its toolkit with more tools, and then extend it with MCP to connect to the HealthOmics genomics platform. You already met the Strands basics in Module 1, so here we focus on growing the agent.

## 🎯 What You'll Learn

1. **Growing an Agent's Toolkit** - Adding more tools to an existing agent
2. **MCP Integration** - Model Context Protocol for tool connectivity
3. **Agent Construction** - Step-by-step agent building process using code
4. **Tool Integration** - Adding custom tools and MCP servers

## 🏗️ What You'll Build

1. **Reconnect to Your One-Tool Agent** - Re-establish the Module 1 Research Assistant with its single `query_variants_by_gene` tool
2. **Grow the Toolkit** - Add two more variant tools plus a workflow monitoring tool
3. **MCP Integration** - Connect to the HealthOmics MCP server for 69+ tools
4. **Experimentation** - Test your agent with real genomics queries

---

## 🛠️ Step 1: Environment Setup

First, let's install the required dependencies and set up our environment.

In [1]:
from IPython.display import HTML

HTML("""
<style>
/* Make all JupyterLab output areas scrollable */
.jp-OutputArea-output {
    max-height: 300px;
    overflow-y: auto;
}
</style>
""")

In [2]:
# Install required dependencies
# Note: You may see dependency conflict warnings from pre-installed SageMaker packages
# (e.g., sparkmagic, shap). These are safe to ignore - they do not affect this workshop.
!pip install "pandas>=2.0,<3.0" --upgrade --quiet
!pip install strands-agents boto3 uv --quiet

# Set up environment for tool consent bypass (for automated workflows)
import os
os.environ["BYPASS_TOOL_CONSENT"] = "true"

print("✅ Environment setup complete!")

✅ Environment setup complete!


## 🔬 Step 2: Reconnect to Your Module 1 Research Assistant

In Module 1 you built a Research Assistant with ONE tool, `query_variants_by_gene`. Let's quickly re-establish it here, then grow it.

First, let's load the variant data (MAF file) from the HealthOmics pipeline output. This is the same data your Module 1 tool queried:


In [3]:
import boto3
import pandas as pd
from io import StringIO

# Get the results bucket and find the MAF file
s3 = boto3.client('s3')
omics = boto3.client('omics')
cfn = boto3.client('cloudformation')

# Get bucket name from CloudFormation stack
stack = cfn.describe_stacks(StackName='infrastructure')['Stacks'][0]
results_bucket = [o['OutputValue'] for o in stack['Outputs'] if o['OutputKey'] == 'WorkflowResultsBucket'][0]

# Find the MAF file from the completed workflow run
runs = omics.list_runs()
for run in runs['items']:
    if run.get('status') == 'COMPLETED':
        run_id = run['id']
        response = s3.list_objects_v2(Bucket=results_bucket, Prefix=f"omics_output/{run_id}/")
        for obj in response.get('Contents', []):
            if obj['Key'].endswith('.maf'):
                maf_key = obj['Key']
                break
        break

obj = s3.get_object(Bucket=results_bucket, Key=maf_key)
maf_df = pd.read_csv(StringIO(obj['Body'].read().decode('utf-8')), sep='\t', comment='#')
print(f"✅ Loaded {len(maf_df)} variants from {maf_key}")
print(maf_df[['Hugo_Symbol', 'Variant_Classification', 'HGVSp_Short']].head(10))


✅ Loaded 41 variants from omics_output/5820471/out/maf_file/output.maf
  Hugo_Symbol Variant_Classification HGVSp_Short
0        UBR4      Missense_Mutation    p.E2097K
1        MTF1      Missense_Mutation     p.S508P
2      LINGO4      Missense_Mutation     p.D513N
3      CCDC93      Missense_Mutation     p.P440S
4        CPS1      Missense_Mutation     p.T166I
5       PARP3      Missense_Mutation     p.E200Q
6       PRDM9                 Silent     p.D201=
7      TRIM39                 Silent     p.R374=
8     HLA-DRA      Missense_Mutation     p.Y175C
9        HEY2      Missense_Mutation     p.A247T


## 🔧 Step 3: Bring Back the Module 1 Tool

Now let's re-establish the single tool from Module 1. Any Python function annotated with `@tool` is discoverable by the agent. Agents autonomously select when to run a tool based on the signature and description of the tool. The agent will determine what arguments to send to the tool and what to do with the response.

Here is `query_variants_by_gene`, exactly as you defined it in Module 1:

**Why redefine it?** Each notebook runs in its own fresh Python session, so the `query_variants_by_gene` function you wrote in Module 1 is not carried over. Even if you went straight from Module 1 to Module 2, this new notebook starts with an empty workspace. Redefining the tool here is how we bring it back. In real projects you would avoid this repetition by saving shared tools in a `.py` file and importing them, which is exactly what you will do in Module 3.

In [4]:
from strands import tool

@tool
def query_variants_by_gene(gene_symbol: str) -> str:
    """Query the somatic variant dataset for all variants in a given gene."""
    results = maf_df[maf_df['Hugo_Symbol'].str.upper() == gene_symbol.upper()]
    if results.empty:
        return f"No variants found for gene {gene_symbol} in this sample."
    cols = ['Hugo_Symbol', 'Variant_Classification', 'HGVSp_Short', 'Chromosome', 'Start_Position', 't_depth', 't_alt_count']
    available_cols = [c for c in cols if c in results.columns]
    summary = results[available_cols].to_string(index=False)
    return f"Found {len(results)} variant(s) in {gene_symbol}:\n{summary}"

print("✅ Module 1 tool is back: query_variants_by_gene")


✅ Module 1 tool is back: query_variants_by_gene


The two most important parts of any Agent are the `model` which provides the intelligence and the `system_prompt` which provides some context. Changing the `system_prompt` is a useful way to specialise an agent and nudge it towards your preferred way of doing things. Let's re-create the Module 1 Research Assistant with its single tool and the genomics system prompt, then run one quick query to confirm it is re-established.

In [5]:
from strands import Agent
from strands.models import BedrockModel

# Re-establish the Module 1 Research Assistant with its single tool
research_assistant = Agent(
    name="Genomics Research Assistant",
    model=BedrockModel(
        model_id="global.anthropic.claude-sonnet-5",
        # 💡 Try Opus 5.5 for deeper reasoning (slower, higher cost): "global.anthropic.claude-opus-5-5"
    ),
    tools=[query_variants_by_gene],
    system_prompt="""You are a genomics research assistant helping researchers interpret somatic variant data from cancer samples.
Use the available tools to answer questions about variants. Always explain findings in plain language."""
)

# Quick confirming query - the Module 1 agent is re-established
response = research_assistant("What variants are in TP53?")
print("🤖 Research Assistant Response:")
print(response)
print(f"\n📊 Tools used: {list(response.metrics.tool_metrics.keys())}")



Tool #1: query_variants_by_gene
Here's what I found in the dataset for **TP53**:

## Summary: 1 variant identified

| Feature | Detail |
|---|---|
| **Variant Type** | Nonsense Mutation |
| **Protein Change** | p.R213* |
| **Location** | Chromosome 17, position 7,674,894 |
| **Sequencing Depth** | 38 total reads |
| **Variant Reads** | 20 reads supporting the mutation (~53% allele frequency) |

### Plain-language interpretation:

- **Nonsense mutation**: This means the mutation introduces a **premature stop codon** at amino acid position 213 (hence "R213\*" — the arginine (R) normally found there is replaced by a stop signal).
- **Functional impact**: This type of mutation typically produces a **truncated, non-functional p53 protein**, since translation halts early and the rest of the protein (which includes critical DNA-binding domains) is never made.
- **Clinical relevance**: TP53 is a major tumor suppressor gene often called the "guardian of the genome." Loss-of-function mutations 

## 🔧 Step 4: Grow the Toolkit

Your Module 1 agent can look up variants by gene, and that is all it can do. Let's grow its toolkit by adding two more tools. Just like before, each new capability is a plain Python function with the `@tool` decorator.

We'll add `get_variant_summary`, which returns summary statistics for the whole dataset, and `get_high_impact_variants`, which filters for the most consequential variant types:

In [6]:
@tool
def get_variant_summary() -> str:
    """Return summary statistics of the variant dataset."""
    total = len(maf_df)
    by_class = maf_df['Variant_Classification'].value_counts().head(10).to_dict()
    top_genes = maf_df['Hugo_Symbol'].value_counts().head(10).to_dict()
    summary = f"Total variants: {total}\n\nVariants by classification:\n"
    for k, v in by_class.items():
        summary += f"  {k}: {v}\n"
    summary += "\nTop 10 mutated genes:\n"
    for k, v in top_genes.items():
        summary += f"  {k}: {v}\n"
    return summary

@tool
def get_high_impact_variants() -> str:
    """Filter for high-impact variants (nonsense, frameshift, splice site)."""
    high_impact = ['Nonsense_Mutation', 'Frame_Shift_Del', 'Frame_Shift_Ins', 'Splice_Site', 'Splice_Region']
    results = maf_df[maf_df['Variant_Classification'].isin(high_impact)]
    if results.empty:
        return "No high-impact variants found."
    cols = ['Hugo_Symbol', 'Variant_Classification', 'HGVSp_Short', 'Chromosome', 'Start_Position', 't_depth', 't_alt_count']
    available_cols = [c for c in cols if c in results.columns]
    return f"Found {len(results)} high-impact variant(s):\n{results[available_cols].to_string(index=False)}"

print("✅ Two new tools added: get_variant_summary, get_high_impact_variants")


✅ Two new tools added: get_variant_summary, get_high_impact_variants


Now let's recreate the Research Assistant with the full three-tool toolkit. The agent will decide which tool to use based on your question. We keep the same `model` and `system_prompt`, and simply pass the larger list of tools.

In [7]:
# Recreate the Research Assistant with the full three-tool toolkit
research_assistant = Agent(
    name="Genomics Research Assistant",
    model=BedrockModel(
        model_id="global.anthropic.claude-sonnet-5",
        # 💡 Try Opus 5.5 for deeper reasoning (slower, higher cost): "global.anthropic.claude-opus-5-5"
    ),
    tools=[query_variants_by_gene, get_variant_summary, get_high_impact_variants],
    system_prompt="""You are a genomics research assistant helping researchers interpret somatic variant data from cancer samples.
Use the available tools to answer questions about variants. Always explain findings in plain language."""
)

# Test with a prompt that exercises the new tools
response = research_assistant("Summarize the variants and show me the high-impact ones")
print("🤖 Research Assistant Response:")
print(response)
print(f"\n📊 Tools used: {list(response.metrics.tool_metrics.keys())}")


I'll pull both the overall variant summary and the high-impact variant subset for you.
Tool #1: get_variant_summary

Tool #2: get_high_impact_variants
## Variant Summary

The dataset contains **41 total somatic variants**, broken down by functional classification as follows:

| Classification | Count | Notes |
|---|---|---|
| Missense_Mutation | 29 | Amino acid–changing, most common type here |
| Silent | 9 | No amino acid change (likely benign) |
| Nonsense_Mutation | 1 | Introduces a premature stop codon |
| Frame_Shift_Ins | 1 | Insertion disrupting the reading frame |
| Intron | 1 | Non-coding region variant |

Mutations are spread across many genes, with most genes (e.g., *UBR4*, *MTF1*, *CPS1*, *PRDM9*, *HLA-DRA*, *HEY2*, etc.) each showing only a single variant — no single gene dominates the mutation landscape in this sample set.

## High-Impact Variants (likely functionally disruptive)

Only **2 variants** met the high-impact criteria (nonsense, frameshift, or splice-site mutat

### 💡 What Just Happened?

Your toolkit grew. In Module 1 your Research Assistant had exactly one tool, `query_variants_by_gene`. You just added two more, `get_variant_summary` and `get_high_impact_variants`, giving you a three-tool Research Assistant.

**The pattern stayed the same:**

- Every tool is a plain Python function with the `@tool` decorator
- You did not rewrite the agent, you just handed it a longer list of tools
- The agent chose which tool to call based on your question, and the `Tools used` line confirms it reached for the new ones

Growing an agent is as simple as writing another Python function and adding it to the list. Next you will add a workflow monitoring tool, and then connect to an MCP server to jump from 3 tools to 69+.

---


## 🔧 Step 5: Adding Custom Tools - Workflow Monitoring

Your Research Assistant can query variant data. But what if you also want to check on your HealthOmics workflow runs? Let's add another tool, this time for monitoring workflows.

The code below shows how easy it is to extend an agent's capabilities with Strands. Any Python function decorated with `@tool` becomes available to the agent.

We'll start with a simple tool for workflow monitoring. The agent can call this tool when a workflow starts to wait on it for a specified time. As you can see this tool doesn't really wait because the workflow has been pre-run. The code in the next cell defines this tool inline, so you can read exactly how it works.

In [8]:
import time
import boto3
from botocore.exceptions import ClientError
from strands import tool

@tool
def wait_for_workflow(run_id: str, max_wait_minutes: int = 60) -> str:
    """
    Wait for a HealthOmics workflow run to complete by polling the run status.
    Checks the run status every 30 seconds and returns early when the run reaches a terminal state.
    
    Args:
        run_id: The HealthOmics run ID to monitor
        max_wait_minutes: Maximum time to wait in minutes (default: 60, max: 120 for safety)
    
    Returns:
        str: Status message indicating the final run state and duration
    """
    # Limit maximum wait time for safety
    max_wait_minutes = min(max(max_wait_minutes, 1), 120)
    
    print(f"🔍 Monitoring HealthOmics run {run_id} (max wait: {max_wait_minutes} minutes)...")
    
    # A real implementation would check the status and return if complete (or failed) and sleep if not.

    omics_client = boto3.client('omics')
    return omics_client.get_run(id=run_id)



Now that our tool is defined, let's create an agent that combines all four tools, the three variant query tools from your Research Assistant plus the new workflow monitoring tool. The agent will decide which tool to use based on your question.

The two most important parts of any Agent are the `model` which provides the intelligence and the `system_prompt` which provides some context. Changing the `system_prompt` is a useful way to specialize an agent and nudge it towards your preferred way of doing things.

In [16]:
# Extend the Research Assistant with the workflow monitoring tool
# Now it can query variants AND monitor workflow runs
research_assistant_with_workflows = Agent(
    name="Genomics Research Assistant + Workflow Monitor",
    description="A research assistant that can query variant data and monitor HealthOmics workflow runs",
    model=BedrockModel(
        model_id="global.anthropic.claude-sonnet-5",
        # 💡 Try Opus 5.5 for deeper reasoning (slower, higher cost): "global.anthropic.claude-opus-5-5"
        max_tokens=4096
    ),
    tools=[query_variants_by_gene, get_variant_summary, get_high_impact_variants, wait_for_workflow],
    system_prompt="""You are a genomics research assistant helping researchers interpret somatic variant data 
and monitor HealthOmics workflow runs. Use the available tools to answer questions about variants 
and check workflow status. Always explain findings in plain language."""
)


/tmp/ipykernel_866406/1204925098.py:6: UserWarning: Invalid configuration parameters: ['max_token'].
Valid parameters are: ['additional_args', 'additional_request_fields', 'additional_response_field_paths', 'cache_config', 'cache_prompt', 'cache_tools', 'context_window_limit', 'guardrail_id', 'guardrail_latest_message', 'guardrail_redact_input', 'guardrail_redact_input_message', 'guardrail_redact_output', 'guardrail_redact_output_message', 'guardrail_stream_processing_mode', 'guardrail_trace', 'guardrail_version', 'include_tool_result_status', 'max_tokens', 'model_id', 'service_tier', 'stop_sequences', 'streaming', 'strict_tools', 'temperature', 'top_p', 'use_native_token_count'].

See https://github.com/strands-agents/harness-sdk/issues/815
  model=BedrockModel(
INFO:botocore.credentials:Found credentials from IAM Role: BaseNotebookInstanceEc2InstanceRole


Lets find out the latest HealthOmics workflow run using the AWS SDK. We will use this to test the agent.

In [10]:
omics_client = boto3.client('omics')
latest_run_id = omics_client.list_runs()['items'][0]['id']
print(latest_run_id)

5820471


Now that we have an example workflow run ID, lets test the newly created agent with an example prompt. As the run has already completed the Agent should be able to determine that status.

In [11]:
# Test the agent with tools
response = research_assistant_with_workflows(f"Can you monitor workflow run {latest_run_id} for me?")
print("🤖 Agent with Tools Response:")
print(response)

# Verify which tool was used
print(f"\n📊 Tools used: {list(response.metrics.tool_metrics.keys())}")

I'll start monitoring that workflow run for you.
Tool #1: wait_for_workflow
🔍 Monitoring HealthOmics run 5820471 (max wait: 60 minutes)...
Good news — the workflow run has already finished successfully! Here's the summary:

**Run 5820471: `genomics-ai-workshop-mutect-run`**
- **Status:** ✅ COMPLETED
- **Workflow type:** Private (Mutect2 tumor-normal somatic variant calling + MAF conversion)
- **Started:** 2026-10-26 19:06:06 UTC
- **Finished:** 2026-10-26 19:40:00 UTC
- **Duration:** ~34 minutes

**Pipeline details:**
- **Tumor sample:** SRR2089363
- **Normal sample:** SRR2089364
- **Reference genome:** hg38 (Broad Institute reference bundle)
- **Tool used:** GATK4 Mutect2 (v4.6.2.0)
- **Scatter count:** 25 (parallelized across genome intervals)

**Output location:**
- MAF file: `s3://aws-genomics-static-us-east-1/omics-data/tumor-normal/maf/test_civic.maf`
- Full run outputs: `s3://infrastructure-workflowresultsbucket-kgqjpdgtzqgz/omics_output/5820471`

The run completed without error

## 🌐 Step 6: MCP Integration - From 3 Tools to 69+ by Connecting to External Tool Servers

So far, your agent has custom tools you wrote in Python. But what if you could connect it to an entire platform of pre-built tools without writing any more code?

That's what Model Context Protocol (MCP) does. MCP is a standard way to connect agents to external tool servers. Think of it like USB-C for AI agents - a universal connector that lets any agent talk to any tool server using a consistent interface built for agent workflows. Instead of writing custom code for every integration, you plug into an MCP server and the tools just appear.

Here's the difference:
- **Without MCP**: You want 10 new tools? Write 10 Python functions, one for each, just like you did when you grew the toolkit.
- **With MCP**: You connect to one server and get all its tools instantly. The HealthOmics MCP server gives you 69+ tools in a single connection.

Adding a tool in Modules 1 and 2 meant writing one Python function each. MCP gives you 69+ tools at once by connecting to a single server, no extra code required.

We'll connect to the AWS HealthOmics MCP server, which provides tools for workflow management, data discovery, validation, and more. Now let's integrate with a tool set provided by an MCP (Model Context Protocol) server connected to the AWS HealthOmics MCP server. First, we create the client which fetches the MCP server from and runs it using `uv`. MCP servers can be remote or local and can communicate on different protocols. For simplicity (and lower latency) we deploy the agent locally and managed by the agent communicating via STDIO:


In [12]:
from strands.tools.mcp import MCPClient
from mcp import stdio_client, StdioServerParameters
import logging

# Configure logging to see what's happening
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


def create_mcp_client():
    """Create an mcp client for the HealthOmics server"""
    
    # Setup MCP client for AWS HealthOmics
    mcp_client = MCPClient(
        lambda: stdio_client(
            StdioServerParameters(
                command="uvx",
                args=["awslabs.aws-healthomics-mcp-server@latest"]            ),
        )
    )

    return mcp_client

# Create MCP client
print("🔌 Setting up MCP connection to HealthOmics server...")
mcp_client = create_mcp_client()
print("✅ MCP client created!")

🔌 Setting up MCP connection to HealthOmics server...
✅ MCP client created!


Now, lets inspect the MCP client to see what tools the server has made available. We will also create an agent and ask it some questions that will require it to use the MCP tools to answer.

In [13]:
# Use the MCP client to discover available tools and create a persistent agent
with mcp_client:
    # Get the tools from the MCP server
    mcp_tools = mcp_client.list_tools_sync()
    print(f"🛠️ Found {len(mcp_tools)} MCP tools from HealthOmics server:")
    
    # Show first few tools
    for i, tool in enumerate(mcp_tools[:5]):
        print(f"  {i+1}. {tool.tool_name}")
    
    if len(mcp_tools) > 5:
        print(f"  ... and {len(mcp_tools) - 5} more tools")

# Store tools and MCP client for reuse in later cells
healthomics_mcp_tools = mcp_tools
healthomics_mcp_client = mcp_client

# Create agent with Research Assistant tools, custom tools, and MCP tools
healthomics_agent = Agent(
    name="Genomics Research Assistant + HealthOmics Platform",
    description="""I'm your genomics research assistant with full HealthOmics platform access. I can help you:
    
    - Query and interpret somatic variant data from your cancer samples
    - Find high-impact variants and summarise your dataset
    - Create and manage HealthOmics workflows
    - Start and monitor workflow runs with real-time status monitoring
    - Analyze workflow performance and optimize resource usage
    - Troubleshoot failed workflows
    - Lint and validate workflow definitions (WDL/CWL)
    
    I have access to your variant data, custom workflow tools, and comprehensive 
    HealthOmics APIs through MCP.""",
    model=BedrockModel(
        model_id="global.anthropic.claude-sonnet-5",
        # 💡 Try Opus 5.5 for deeper reasoning (slower, higher cost): "global.anthropic.claude-opus-5-5"
    ),
    tools=healthomics_mcp_tools + [wait_for_workflow, query_variants_by_gene, get_variant_summary, get_high_impact_variants],
    system_prompt="""
    You are a genomics research assistant and AWS HealthOmics workflow orchestrator. Your role is to:

    1. Query and interpret somatic variant data from cancer samples using your variant query tools
    2. Help users create, deploy, and manage HealthOmics workflows
    3. Monitor workflow execution and provide real-time status updates
    4. Analyze workflow performance and suggest optimizations
    5. Troubleshoot workflow failures with detailed diagnostics
    6. Validate workflow definitions and ensure best practices
    
    Always provide clear, actionable guidance and use the HealthOmics MCP tools
    to perform operations. Be proactive in suggesting optimizations and best practices.

    When creating workflows, follow these guidelines:
    1. Unless otherwise instructed, create workflows in WDL 1.1
    2. Ensure all tasks have suitable cpu, memory and container directives favoring use of containers from quay.io
    3. Use the LintAHOWorkflowDefinition or LintAHOWorkflowBundle tools to ensure correctness
    4. When possible and logical, scatter over inputs or genomic intervals to improve computational efficiency
    5. When creating updates to existing workflows, create workflow versions rather than new workflows
    6. Use `set -euo pipefail` in task commands
    7. `echo` the names and values of task inputs in the task command to assist with debugging
    
    IMPORTANT: When you start a workflow run, you will receive a run ID. Use the wait_for_workflow tool 
    with this run ID to monitor the run status until completion. This tool polls the HealthOmics API 
    every 30 seconds and returns early when the run reaches a terminal state.
    """
)

print("\n🤖 Testing the full agent with MCP tools...")

# Test the agent with a real HealthOmics query using the MCP client context
with healthomics_mcp_client:
    response = healthomics_agent("What regions is HealthOmics available in and what workflows are available in my account?")
    print("\n📋 Agent Response:")
    print(response)
    
    print(f"\n📊 Execution Metrics:")
    print(f"  • Total tokens: {response.metrics.accumulated_usage.get('totalTokens', 0)}")
    print(f"  • Execution time: {sum(response.metrics.cycle_durations):.2f} seconds")
    print(f"  • Tools used: {list(response.metrics.tool_metrics.keys())}")

print("\n✅ HealthOmics agent is now available for experimentation in later cells!")
print("   Use 'healthomics_agent' variable to interact with the agent")
print("   Remember to use 'with healthomics_mcp_client:' context for MCP tool access")

🛠️ Found 77 MCP tools from HealthOmics server:
  1. ListAHOWorkflows
  2. CreateAHOWorkflow
  3. GetAHOWorkflow
  4. CreateAHOWorkflowVersion
  5. ListAHOWorkflowVersions
  ... and 72 more tools


INFO:botocore.credentials:Found credentials from IAM Role: BaseNotebookInstanceEc2InstanceRole



🤖 Testing the full agent with MCP tools...

Tool #1: GetAHOSupportedRegions

Tool #2: ListAHOWorkflows
## AWS HealthOmics Availability & Your Workflows

### Supported Regions (10 total)
AWS HealthOmics is currently available in:

| Americas | Europe | Asia Pacific | Middle East |
|---|---|---|---|
| us-east-1 (N. Virginia) | eu-west-1 (Ireland) | ap-northeast-1 (Tokyo) | il-central-1 (Israel) |
| us-east-2 (Ohio) | eu-west-2 (London) | ap-northeast-2 (Seoul) | |
| us-west-2 (Oregon) | eu-central-1 (Frankfurt) | ap-southeast-1 (Singapore) | |

### Your Private Workflows (us-east-1)
You currently have **1 private workflow**:

| Name | Workflow ID | Status | Type | Created |
|---|---|---|---|---|
| **genomics-ai-workshop-mutect2** | `6830174` | ✅ ACTIVE | PRIVATE | 2026-10-26 |

- **ARN:** `arn:aws:omics:us-east-1:424159461804:workflow/6830174`
- No description currently set
- No storage type/capacity configured at the workflow level (likely configured per-run or uses workflow version de

## 🧪 Step 7: Experiment with Your Agent

Your agent now has both your custom Research Assistant tools AND 69+ MCP tools from HealthOmics. Try mixing and matching, ask about your variant data, your workflows, or both!

### 💡 **Suggested Prompts to Try:**

**Variant Queries (your @tool functions):**
- "What variants are in TP53 and what do they mean for this tumour?"
- "Give me a summary of all variants and highlight anything concerning"
- "Are there any high-impact variants? Which genes are affected?"

**Workflow & Platform Queries (MCP tools):**
- "Find a workflow in my account related to somatic variant calling, show me the details"
- "List all workflow runs from the past week"
- "What are the best practices for genomics workflows in HealthOmics?"

**Combined Queries (agent uses multiple tools):**
- "Summarise my variant results and check if the workflow that produced them completed successfully"
- "Create a simple hello-world workflow in WDL, validate it, then show me the definition"

### 🔧 **How to Use:**
1. Modify the prompt in the cell below
2. Run the cell to see the agent's response
3. Try different prompts to explore the agent's capabilities
4. The agent will automatically choose the right tool based on your question


In [14]:
# 🧪 EXPERIMENT CELL - Try different prompts here!
# Change the prompt below to experiment with different questions

experiment_prompt = "What variants are in TP53? Are any of them high-impact?"

print(f"🔍 Asking agent: {experiment_prompt}")
print("=" * 60)

# Use the agent with MCP client context
with healthomics_mcp_client:
    response = healthomics_agent(experiment_prompt)
    print(response)
    
    print(f"\n📊 Execution Metrics:")
    print(f"  • Total tokens: {response.metrics.accumulated_usage.get('totalTokens', 0)}")
    print(f"  • Execution time: {sum(response.metrics.cycle_durations):.2f} seconds")
    print(f"  • Tools used: {list(response.metrics.tool_metrics.keys())}")

🔍 Asking agent: What variants are in TP53? Are any of them high-impact?

Tool #3: query_variants_by_gene

Tool #4: get_high_impact_variants
## TP53 Variant Analysis

### Variant Found in TP53

| Gene | Classification | Protein Change | Location | Depth | Alt Reads | VAF |
|---|---|---|---|---|---|---|
| **TP53** | Nonsense_Mutation | **p.R213*** | chr17:7,674,894 | 38 | 20 | **~52.6%** |

### Is it High-Impact? **Yes ✅**

This variant appears in the high-impact variant list. Here's why it matters:

- **Nonsense mutation (p.R213*)**: This substitution introduces a premature stop codon at amino acid position 213, truncating the TP53 protein before it reaches its full length (393 aa). This almost certainly destroys the DNA-binding domain function of p53, a critical tumor suppressor.
- **Variant Allele Frequency (VAF) ~53%**: With 20 of 38 reads supporting the alternate allele, this suggests the mutation is present in roughly half the sequenced alleles — consistent with a heterozygous muta

In [15]:
# 🧪 EXPERIMENT CELL 2 - Try another prompt!
# This cell is ready for your next experiment

experiment_prompt_2 = """
  Create a simple WDL workflow that says hello to genomics researchers and display it in your response.
  Check that the workflow syntax is correct, then create it in HealthOmics.
  """

print(f"🔍 Asking agent: {experiment_prompt_2}")
print("=" * 60)

with healthomics_mcp_client:
    response = healthomics_agent(experiment_prompt_2)
    print(response)
    
    print(f"\n📊 Execution Metrics:")
    print(f"  • Total tokens: {response.metrics.accumulated_usage.get('totalTokens', 0)}")
    print(f"  • Execution time: {sum(response.metrics.cycle_durations):.2f} seconds")
    print(f"  • Tools used: {list(response.metrics.tool_metrics.keys())}")

🔍 Asking agent: 
  Create a simple WDL workflow that says hello to genomics researchers and display it in your response.
  Check that the workflow syntax is correct, then create it in HealthOmics.
  
I'll create a simple "Hello Genomics Researchers" WDL workflow, lint it for correctness, then package and deploy it to HealthOmics.

## The Workflow Definition

```wdl
version 1.1

workflow HelloGenomicsResearchers {
  meta {
    description: "A simple workflow that greets genomics researchers."
  }

  input {
    String name = "Genomics Researcher"
  }

  call SayHello {
    input:
      name = name
  }

  output {
    File   greeting_file = SayHello.greeting_file
    String greeting_text = SayHello.greeting_text
  }
}

task SayHello {
  input {
    String name
  }

  command <<<
    set -euo pipefail
    echo "Task input - name: ~{name}"
    echo "Hello, ~{name}! Welcome to the world of genomics research." > greeting.txt
    cat greeting.txt
  >>>

  output {
    File   greeting_file = "

INFO:strands.event_loop._recover_message_on_max_tokens_reached:handling max_tokens stop reason - replacing all tool uses with error messages


MaxTokensReachedException: Model stopped generating due to maximum token limit. The partial message has been added to the conversation history. You can continue by calling the agent again. For more information see: https://strandsagents.com/docs/user-guide/concepts/agents/agent-loop/#maxtokensreachedexception

In [ ]:
# 🧪 EXPERIMENT CELL 3 - Add your own prompt!
# Copy this cell or modify it to try more experiments

# TODO: Replace this with your own prompt
your_prompt = "Summarise my variant data and then check the status of my most recent workflow run"

print(f"🔍 Your experiment: {your_prompt}")
print("=" * 60)

with healthomics_mcp_client:
    response = healthomics_agent(your_prompt)
    print(response)
    
    print(f"\n📊 Execution Metrics:")
    print(f"  • Total tokens: {response.metrics.accumulated_usage.get('totalTokens', 0)}")
    print(f"  • Execution time: {sum(response.metrics.cycle_durations):.2f} seconds")
    print(f"  • Tools used: {list(response.metrics.tool_metrics.keys())}")

### 🎯 **What You Can Explore:**

**Your Research Assistant Tools:**
- Query variants by gene (TP53, BRCA1, EGFR, etc.)
- Get variant summaries and statistics
- Find high-impact variants (nonsense, frameshift, splice site)

**System Prompt**
- Modify the system prompt to change the agent's behaviour, try giving it a personality or asking it to format results differently

**Workflow Management (MCP):**
- List, create, and get details about workflows
- Create workflow versions
- Validate and lint workflow definitions

**Workflow Execution (MCP):**
- Start workflow runs
- Monitor run status and progress
- Get run details and logs

**Data Management (MCP):**
- Work with reference stores and sequence stores
- Import and export genomics data

**System Information (MCP):**
- Check supported regions
- Get service quotas and limits

Feel free to ask complex questions. The agent will use multiple tools if needed to provide comprehensive answers!


## 🎓 Summary: What You've Built

Congratulations! You've progressed from a one-tool code-first agent (Module 1) to a fully code-driven agent with MCP integration. Here's what you accomplished:

### 🏗️ The Build-Up
1. **Reconnected Your One-Tool Agent**: Re-established the Module 1 Research Assistant with its single `query_variants_by_gene` tool
2. **Grew the Toolkit**: Added `get_variant_summary` and `get_high_impact_variants` to reach a three-tool Research Assistant, all in code with the `@tool` decorator
3. **Workflow Monitor**: Extended with a workflow monitoring tool
4. **MCP-Powered Agent**: Connected to the HealthOmics MCP server for 69+ tools

### 💡 Key Insight
You started Module 1 with 1 tool and grew to 3 in code; connecting one MCP server took it to 69+. That's the power of code + open protocols.

### ➡️ Coming Up in Module 3
Your Research Assistant is about to join a team. In the next module, you'll see it become one of 5 specialised agents, coordinated by a supervisor that routes questions to the right expert.
